In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))

from dotenv import load_dotenv
from openagv import Project
from openagv.llm import LLMConfig
from openagv.storage import LocalStorageBackend

# Load environment variables
load_dotenv()

# Create project with LLM config for the executor (tool-calling model)
project = Project(
    name="Flowers with Overlays",
    llm_config=LLMConfig(provider="ollama", model="qwen2.5:14b"),
    storage=LocalStorageBackend(root="./project_data", project_id="sample3"),
    width=1920,
    height=1080,
    fps=30.0,
)

print(f"Project '{project.name}' created (id: {project.id[:8]}...)")

Project 'Flowers with Overlays' created (id: 1f3eb038...)


In [ ]:
from openagv.llm import create_clients
from openagv.modules.vision import ORVisionAnalyzer
from openagv.modules.textcard import TextCardGenerator

# Vision model needs its own client (separate from the executor model)
vision_client, _, _ = create_clients("ollama", "llama3.2-vision")

# Register modules on the project
project.register_module(ORVisionAnalyzer(client=vision_client, model="llama3.2-vision"))
project.register_module(TextCardGenerator(output_dir="./text_cards", width=1920, height=1080, font_size=36))

# Add flower images
assets = project.add_assets("../assets/*.jpg")
print(f"Added {len(assets)} assets")

# Submit the job — runs autonomously in background
job = project.submit(
    """Create a video showcasing flowers with descriptive text overlays.
    
    For each flower image:
    1. First analyze the image to get a description
    2. Generate a lower-third text overlay with the flower name and a short description
    3. Add the flower to the main timeline track (3 seconds each)
    4. Add the text overlay at the same time position on an overlay track
    
    Make sure every flower has both a background clip and a text overlay.
    Sort flowers by how vibrant/colorful they appear.""",
    author="user:notebook",
    debug=True,
)

print(f"Job {job.id[:8]}... submitted (status: {job.status.value})")
await job.wait()
print(f"Job finished (status: {job.status.value})")

Added 6 assets
Job 1a1a3f38... submitted (status: 2)
[INFO] Starting execution...
[DEBUG] Chat History: 2 messages


In [ ]:
# Inspect timeline state via project
print(project.timeline.get_summary())
print(f"\nOverlay tracks: {project.timeline.get_overlay_tracks()}")
print(f"\nJob chat history ({len(job.chat_history)} messages):")
for msg in job.chat_history:
    print(f"  [{msg.role}] ({msg.author}): {msg.content[:80]}..."
          if len(msg.content) > 80 else f"  [{msg.role}] ({msg.author}): {msg.content}")

In [ ]:
# Submit a follow-up job on the same project
# (the previous job is done, so we can submit a new one)
followup = project.submit(
    "Please verify that every flower in the timeline has a corresponding text overlay. "
    "If any are missing, add them now.",
    author="user:notebook",
    debug=True,
)
await followup.wait()
print(f"Follow-up finished (status: {followup.status.value})")
print(project.timeline.get_summary())

In [ ]:
# Export timeline as OTIO file
project.timeline.to_otio_file("flowers_overlay.otio")
print("Timeline saved to flowers_overlay.otio")

In [ ]:
# Render using the project's storage backend
from openagv.renderer import FfmpegOTIORenderer

renderer = FfmpegOTIORenderer(storage=project.storage)
renderer.set_otio(project.timeline)
renderer.validate()
renderer.render("flowers_with_overlays.mp4", store_key="renders/flowers.mp4")

In [ ]:
# Serialize the entire project to JSON (for storage in JSONB, etc.)
import json

project_data = project.to_dict()
print(f"Project serialized: {len(json.dumps(project_data))} chars")
print(f"  Assets: {len(project_data['asset_bin']['assets'])}")
print(f"  Jobs: {len(project_data['jobs'])}")
print(f"  Modules: {project_data['modules']}")
print(f"  Timeline tracks: {len(project_data['timeline']['_otio'].get('tracks', {}).get('children', []))}")

# Round-trip: deserialize back (api_key re-injected at load time)
from openagv import Project
restored = Project.from_dict(project_data, storage=project.storage)
print(f"\nRestored project '{restored.name}' with {len(restored.asset_bin.assets)} assets")
print(f"Timeline: {restored.timeline.get_summary()}")